<a href="https://colab.research.google.com/github/BrajanNieto/DeepLearning/blob/main/Grupo4_Laboratorio01_PatchTST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from pandas.tseries.offsets import DateOffset

In [ ]:
class StandardScaler:
    def __init__(self, mean, std, eps=1e-6):
        self.mean = mean.astype(np.float32)
        self.std = np.where(std < eps, eps, std).astype(np.float32)
        self.eps = eps

    def transform(self, x):
        return (x - self.mean) / self.std

    def inverse_transform(self, x):
        return x * self.std + self.mean

In [ ]:
# Calcular las medias y desviaciones por archivo
def make_scalers(train_segments, feature_cols):
    scalers = []
    for i, seg in enumerate(train_segments):
        tr   = seg[feature_cols].dropna()
        mean = tr.mean(axis=0).to_numpy(dtype=np.float32)   # orden = feature_cols
        std  = tr.std(axis=0, ddof=0).to_numpy(dtype=np.float32)
        scalers.append(StandardScaler(mean, std))
    return scalers

In [ ]:
class ETTDataset(Dataset):
    def __init__(
        self,
        segments, # Lista de segmentos temporales (deberian ser dos)
        seq_len=96,
        pred_len=96,
        target_col="OT",
        feature_cols=None,
        scalers=None, # Creamos un Scaler por archivo (son dos)
        split_name="train", # "train", "val", "test"
    ):
        if feature_cols is None:
            feature_cols = list(segments[0].columns)

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.target_col = target_col
        self.feature_cols = feature_cols
        self.split_name = split_name

        self.series_data = []
        self.scalers = scalers
        self.index_map = []
        self.target_idx = self.feature_cols.index(self.target_col)

        # Preparar los datos
        for sidx, seg in enumerate(segments):
            seg = seg.sort_index()
            miss = [c for c in self.feature_cols if c not in seg.columns]
            if miss:
                raise ValueError(f"Serie {sidx}: faltan columnas {miss}.")

            seg = seg[self.feature_cols].copy().dropna()

            scaler = self.scalers[sidx]
            X_all = scaler.transform(seg.to_numpy(dtype=np.float32))  # [n_seq, n_features]
            y_all = X_all[:, self.target_idx]                         # [n_seq]

            T = X_all.shape[0]
            max_start = T - (self.seq_len + self.pred_len)
            if max_start < 0:
                # No alcanza para la ventana
                starts = np.array([], dtype=np.int64)
            else:
                starts = np.arange(0, max_start + 1, dtype=np.int64)

            self.series_data.append({"X": X_all, "y": y_all, "starts": starts})

        # Indice
        for sidx in (0, 1):
            for st in self.series_data[sidx]["starts"]:
                self.index_map.append((sidx, int(st)))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        sidx, st = self.index_map[idx]
        X_all = self.series_data[sidx]["X"]
        y_all = self.series_data[sidx]["y"]

        x = X_all[st : st + self.seq_len, :]
        y = y_all[st + self.seq_len : st + self.seq_len + self.pred_len]

        x = torch.from_numpy(x).float()
        y = torch.from_numpy(y).float().unsqueeze(-1)
        series_id = torch.tensor(sidx, dtype=torch.long)

        return x, y, series_id

In [ ]:
def read_ETT_csv(path):
    df = pd.read_csv(path)
    if "date" not in df.columns:
        raise ValueError(f"{path} no tiene columna 'date'.")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True).set_index("date")
    return df

In [ ]:
def split_TrainValTest(df):
    start = df.index.min()
    train_end = start + DateOffset(months=12)
    val_end   = train_end + DateOffset(months=4)
    test_end  = val_end + DateOffset(months=4)
    return (
        df.loc[(df.index >= start) & (df.index < train_end)].copy(),
        df.loc[(df.index >= train_end) & (df.index < val_end)].copy(),
        df.loc[(df.index >= val_end) & (df.index < test_end)].copy(),
    )

In [ ]:
def make_DataLoader(df0,df1,n_batch=64):
    df0_train, df0_val, df0_test = split_TrainValTest(df0)
    df1_train, df1_val, df1_test = split_TrainValTest(df1)

    scalers = make_scalers([df0_train, df1_train], feature_cols)

    trainDataset = ETTDataset([df0_train, df1_train],
                              seq_len=input_horizon, pred_len=output_horizon,
                              target_col="OT", feature_cols=feature_cols,
                              scalers=scalers, split_name="train")

    valDataset   = ETTDataset([df0_val, df1_val], seq_len=input_horizon,
                              pred_len=output_horizon,
                              target_col="OT", feature_cols=feature_cols,
                              scalers=scalers, split_name="val")

    testDataset  = ETTDataset([df0_test, df1_test], seq_len=input_horizon,
                              pred_len=output_horizon,
                              target_col="OT", feature_cols=feature_cols,
                              scalers=scalers, split_name="test")

    trainLoader = DataLoader(trainDataset, batch_size=n_batch, shuffle=True, drop_last=True)
    valLoader   = DataLoader(  valDataset, batch_size=n_batch, shuffle=False)
    testLoader  = DataLoader( testDataset, batch_size=n_batch, shuffle=False)

    return trainLoader, valLoader, testLoader

## ETTh (1 hora)

In [ ]:
input_horizon  =  96
output_horizon = 192

feature_cols = ["HUFL","HULL","MUFL","MULL","LUFL","LULL","OT"]

In [ ]:
df0 = read_ETT_csv("ETTh1.csv")
df1 = read_ETT_csv("ETTh2.csv")

In [ ]:
trainLoader, valLoader, testLoader = make_DataLoader(df0,df1)

## ETTm (15 min)

In [ ]:
df0 = read_ett_csv("ETTm1.csv")
df1 = read_ett_csv("ETTm2.csv")

In [ ]:
trainLoader, valLoader, testLoader = make_DataLoader(df0,df1)